# Data Cleaning

## Import

In [391]:
import pandas as pd
import numpy as np
import re

pd.set_option('display.float_format', '{:.2f}'.format)

raw_df = pd.read_csv('data/movies.csv', on_bad_lines='skip')
df = raw_df.copy()


## Checking out dataset

In [392]:
print(df.isna().sum())

df[30:200].head(20)


MOVIES         0
YEAR         644
GENRE         80
RATING      1820
ONE-LINE       0
STARS          0
VOTES       1820
RunTime     2958
Gross       9539
dtype: int64


,MOVIES,YEAR,GENRE,RATING,ONE-LINE,STARS,VOTES,RunTime,Gross
30,Fear Street: 1666,(2021),"\nHorror, Mystery",6.70,\nThe origins of Sarah Fier's curse are finall...,\n Director:\nLeigh Janiak\n| \n Stars:\...,"27,447",114.00,NaN
31,Animal Kingdom,(2016– ),"\nCrime, Drama",8.20,"\nCenters on a Southern California family, who...","\n \n Stars:\nShawn Hatosy, \nBe...","26,926",60.00,NaN
32,Brooklyn Nine-Nine,(2013–2022),"\nComedy, Crime",8.40,\nComedy series following the exploits of Det....,"\n \n Stars:\nAndy Samberg, \nSt...","248,583",22.00,NaN
33,NCIS: Naval Criminal Investigative Service,(2003– ),"\nAction, Crime, Drama",7.70,\nThe cases of the Naval Criminal Investigativ...,"\n \n Stars:\nMark Harmon, \nDav...","125,619",60.00,NaN
34,Kingdom,(2019– ),"\nAction, Drama, History",8.40,\nWhile strange rumors about their ill King gr...,"\n \n Stars:\nJu Ji-Hoon, \nBae ...","34,906",45.00,NaN
35,Modern Family,(2009–2020),"\nComedy, Drama, Romance",8.40,\nThree different but related families face tr...,"\n \n Stars:\nEd O'Neill, \nSofí...","378,030",22.00,NaN
36,Titans,(I) (2018– ),"\nAction, Adventure, Crime",7.70,\nA team of young superheroes combat evil and ...,"\n \n Stars:\nBrenton Thwaites, ...","73,656",45.00,NaN
37,The Witcher,(2019– ),"\nAction, Adventure, Fantasy",8.20,"\nGeralt of Rivia, a solitary monster hunter, ...","\n \n Stars:\nHenry Cavill, \nAn...","336,484",60.00,NaN
38,Downton Abbey,(2010–2015),"\nDrama, Romance",8.70,\nA chronicle of the lives of the British aris...,"\n \n Stars:\nHugh Bonneville, \...","171,804",58.00,NaN
39,Shingeki no kyojin,(2013–2022),"\nAnimation, Action, Adventure",9.00,\nAfter his hometown is destroyed and his moth...,"\n \n Stars:\nYûki Kaji, \nMarin...","242,582",24.00,NaN


## Processing

In [393]:
df = df.drop_duplicates()
df = df.rename(columns = str.title)
df = df.rename(columns = {'One-Line' : 'One_Line_Summary'})

## Movies
df['Movies'] = df['Movies'].str.strip()

## Year
df['Year'] = df['Year'].str.strip()
df['Year'] = df['Year'].str.replace(r'[()]', '' ,regex = True)
df['Year'] = df['Year'].str.replace(r'[–—]', '-', regex=True)


df[['Start_Year', 'End_Year']] = df['Year'].str.extract(r'(\d{4})-?(\d{4})?')
df[['Start_Year', 'End_Year']] = df[['Start_Year', 'End_Year']].fillna('Unknown')


df['Season'] = df['Year'].str.extract(r'([I|X|V|L]+)')
df['Season'] = df['Season'].str.strip().fillna('Unknown')


## Genre
df[['Genre1', 'Genre2', 'Genre3']] = df['Genre'].str.strip().str.split(',', expand = True)
df[['Genre1', 'Genre2', 'Genre3']] = df[['Genre1', 'Genre2', 'Genre3']].fillna('None')

##Rating
df['Rating'] = df['Rating'].fillna(0.0)

## Votes
df['Votes'] = pd.to_numeric(df['Votes'], errors = 'coerce')
df['Votes'] = df['Votes'].fillna(0.0)

## Stars
df['Stars'] = df['Stars'].str.replace(r'\n','' ,regex= True)

df['Directors'] = df['Stars'].str.extract(r'Director:\s*([^|]+)').fillna('Unknown')

df['Stars'] = df['Stars'].str.extract(r'Stars:\s*(.+)').fillna('Unknown')

df[['Star_1', 'Star_2', 'Star_3', 'Star_4']] = df['Stars'].str.split(',', n = 3, expand = True)
df[['Star_1', 'Star_2', 'Star_3', 'Star_4']] = df[['Star_1', 'Star_2', 'Star_3', 'Star_4']].apply(lambda col: col.str.strip()).fillna('None')

## One Line Summary
df['One_Line_Summary'] = df['One_Line_Summary'].str.strip()
df['One_Line_Summary'] = df['One_Line_Summary'].str.replace(r'(\n)', '')
df['One_Line_Summary'] = df['One_Line_Summary'].str.replace(r'(Add a Plot)', 'Unknown')

## Gross
df['Gross'] = df['Gross'].str.replace(',', '', regex=False)
df['Gross(Million Dollars)'] = df['Gross'].str.extract(r'(\d+(?:\.\d+)?)').astype(float).round(2).fillna(0.0)

df = df.drop(columns=['Genre', 'Year', 'Stars', 'Gross'])

## Runtime
df['Runtime'] = df['Runtime'].fillna(0.0)



In [394]:
df.head(50)

,Movies,Rating,One_Line_Summary,Votes,Runtime,Start_Year,End_Year,Season,Genre1,Genre2,Genre3,Directors,Star_1,Star_2,Star_3,Star_4,Gross(Million Dollars)
0,Blood Red Sky,6.10,A woman with a mysterious illness is forced in...,0.00,121.00,2021,Unknown,Unknown,Action,Horror,Thriller,Peter Thorwarth,Peri Baumeister,Carl Anton Koch,Alexander Scheer,Kais Setti,0.00
1,Masters of the Universe: Revelation,5.00,The war for Eternia begins again in what may b...,0.00,25.00,2021,Unknown,Unknown,Animation,Action,Adventure,Unknown,Chris Wood,Sarah Michelle Gellar,Lena Headey,Mark Hamill,0.00
2,The Walking Dead,8.20,Sheriff Deputy Rick Grimes wakes up from a com...,0.00,44.00,2010,2022,Unknown,Drama,Horror,Thriller,Unknown,Andrew Lincoln,Norman Reedus,Melissa McBride,Lauren Cohan,0.00
3,Rick and Morty,9.20,An animated series that follows the exploits o...,0.00,23.00,2013,Unknown,Unknown,Animation,Adventure,Comedy,Unknown,Justin Roiland,Chris Parnell,Spencer Grammer,Sarah Chalke,0.00
4,Army of Thieves,0.00,"A prequel, set before the events of Army of th...",0.00,0.00,2021,Unknown,Unknown,Action,Crime,Horror,Matthias Schweighöfer,Matthias Schweighöfer,Nathalie Emmanuel,Ruby O. Fee,Stuart Martin,0.00
5,Outer Banks,7.60,A group of teenagers from the wrong side of th...,0.00,50.00,2020,Unknown,Unknown,Action,Crime,Drama,Unknown,Chase Stokes,Madelyn Cline,Madison Bailey,Jonathan Daviss,0.00
6,The Last Letter from Your Lover,6.80,A pair of interwoven stories set in the past a...,0.00,110.00,2021,Unknown,Unknown,Drama,Romance,None,Augustine Frizzell,Shailene Woodley,Joe Alwyn,Wendy Nottingham,Felicity Jones,0.00
7,Dexter,8.60,"By day, mild-mannered Dexter is a blood-spatte...",0.00,53.00,2006,2013,Unknown,Crime,Drama,Mystery,Unknown,Michael C. Hall,Jennifer Carpenter,David Zayas,James Remar,0.00
8,Never Have I Ever,7.90,The complicated life of a modern-day first gen...,0.00,30.00,2020,Unknown,Unknown,Comedy,None,None,Unknown,Maitreyi Ramakrishnan,Poorna Jagannathan,Darren Barnet,John McEnroe,0.00
9,Virgin River,7.40,"Seeking a fresh start, nurse practitioner Meli...",0.00,44.00,2019,Unknown,Unknown,Drama,Romance,None,Unknown,Alexandra Breckenridge,Martin Henderson,Colin Lawrence,Tim Matheson,0.00
